# Day 24 — Pre-trained Models & Transfer Learning

## 1. Learning Objectives
- Understand the concept of Transfer Learning.
- Load state-of-the-art models (like ResNet) from `torchvision.models`.
- Freeze the pre-trained weights so they don't get destroyed during training.
- Replace the final Fully Connected layer to match your specific dataset.

## 2. Prerequisites
- `torch.nn` (Day 9)
- CNNs (Day 23)
- Autograd / requires_grad (Day 5)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

## 3. Concept Explanation: Transfer Learning
Training a deep CNN from scratch requires millions of images and weeks of GPU time. Google and Meta have already done this for us using the **ImageNet** dataset (1.2 million images, 1000 categories).

**Transfer Learning**: We take their fully trained model, keep all the layers that detect edges, shapes, and textures (Feature Extraction), and only swap out the very last layer to predict our own categories (e.g., Cats vs Dogs instead of 1000 animals).

This allows you to train a highly accurate model on a standard laptop in minutes with very little data.

## 8. Simple Example: Loading ResNet18
`torchvision.models` contains dozens of models. We will use `ResNet18`, a standard and powerful architecture.

In [ ]:
# Load the model WITH the pretrained ImageNet weights
# (Downloading weights might take a few seconds the first time)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Print the last few lines of the architecture
print(resnet.fc)

Notice that the last layer (`resnet.fc`) is an `nn.Linear(in_features=512, out_features=1000)`.
It predicts 1000 classes. If we are building a Cat vs Dog classifier, we need `out_features=2`.

## 9. Code Walkthrough: Fine-Tuning
To adapt ResNet18 to our Cats vs Dogs problem, we must follow 3 steps:
1. Freeze all the existing weights.
2. Overwrite `resnet.fc` with a new, untrained Linear layer.
3. Pass ONLY the new layer's parameters to the Optimizer.

In [ ]:
# Step 1: Freeze all parameters
for param in resnet.parameters():
    param.requires_grad = False

# Step 2: Replace the final layer
# The old layer had 512 in_features. Our new layer must match that.
num_ftrs = resnet.fc.in_features

# When we assign this new layer, PyTorch automatically sets requires_grad=True for it!
resnet.fc = nn.Linear(num_ftrs, 2) 

print("New final layer:", resnet.fc)

# Step 3: Setup the Optimizer
# IMPORTANT: We only pass the parameters that require gradients!
optimizer = torch.optim.Adam(resnet.fc.parameters(), lr=0.001)

print("Transfer Learning setup complete!")

## 11. Practice Exercise 1: VGG16
Import `models.vgg16(weights=models.VGG16_Weights.DEFAULT)`.
Unlike ResNet, VGG16's final layer is named `classifier` and it is an `nn.Sequential` block. The final sub-layer is `classifier[6]`, which has `in_features=4096` and `out_features=1000`.

Freeze the VGG16 model, and replace `classifier[6]` with a Linear layer that predicts 5 classes.

In [ ]:
# Write your code here

In [ ]:
# SOLUTION
vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

for param in vgg.parameters():
    param.requires_grad = False
    
in_feats = vgg.classifier[6].in_features
vgg.classifier[6] = nn.Linear(in_feats, 5)

print(vgg.classifier[6])

## 13. Debugging Challenge
A developer wrote this transfer learning code, but their model isn't learning anything (the loss never decreases). Find the bug.

In [ ]:
dev_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

dev_model.fc = nn.Linear(512, 10)

for param in dev_model.parameters():
    param.requires_grad = False
    
# dev_opt = torch.optim.Adam(dev_model.fc.parameters(), lr=0.001)

**Solution:** They changed the final layer `dev_model.fc = nn.Linear(512, 10)`, but THEN they ran the `for param in dev_model.parameters(): param.requires_grad = False` loop. This froze the entire network, *including their brand new layer*. 
You must freeze the network FIRST, and replace the layer SECOND.

## 17. Interview Questions
1. **Why do we freeze the feature extraction layers during Transfer Learning?**
   *Answer*: The pretrained layers already know how to extract perfect features (edges, textures). Our new final layer is randomly initialized and completely incompetent. If we don't freeze the early layers, the massive gradients caused by the incompetent final layer will propagate backward and destroy the perfectly tuned weights of the early layers.
2. **Can you unfreeze the layers later?**
   *Answer*: Yes! This is called "Fine-Tuning". You train only the final layer for a few epochs until it converges. Then, you unfreeze the entire network (`requires_grad = True`), drop the learning rate significantly, and train for a few more epochs to fine-tune the entire architecture specifically to your dataset.

## 19. Day Summary
- Transfer Learning saves immense time and compute.
- `models.resnet18(weights=models.ResNet18_Weights.DEFAULT)` loads a pre-trained model.
- **Freeze First**: `for p in model.parameters(): p.requires_grad = False`.
- **Replace Second**: `model.fc = nn.Linear(...)`.
- Ensure your optimizer only updates the new parameters.